Tensorflow

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization


import time

Mnist -> fashion

In [16]:
#mnist цифры

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

X_train = x_train.reshape(x_train.shape[0], -1)
X_test = x_test.reshape(x_test.shape[0], -1)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (60000, 784)
X_test shape: (10000, 784)


In [17]:
def build_model(input_shape, neurons1=128, neurons2=64, dropout_rate=0.2):
    model = Sequential([
        Dense(neurons1, activation='relu', input_shape=(input_shape,)),
        BatchNormalization(),
        Dropout(dropout_rate),

        Dense(neurons2, activation='relu'),
        BatchNormalization(),
        Dropout(dropout_rate),

        Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [18]:
model = build_model(X_train.shape[1])

start_time = time.time()

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    verbose=1
)

end_time = time.time()
total_time = end_time - start_time

y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

report = classification_report(y_test, y_pred)
print(f"Metrics: {report}")

print(f"Время обучения: {total_time/60:.2f} минут")

Epoch 1/50
188/188 [==============================] - 3s 11ms/step - loss: 0.5323 - accuracy: 0.8453 - val_loss: 0.2060 - val_accuracy: 0.9387
Epoch 2/50
188/188 [==============================] - 2s 9ms/step - loss: 0.2364 - accuracy: 0.9313 - val_loss: 0.1510 - val_accuracy: 0.9557
Epoch 3/50
188/188 [==============================] - 2s 9ms/step - loss: 0.1776 - accuracy: 0.9462 - val_loss: 0.1304 - val_accuracy: 0.9615
Epoch 4/50
188/188 [==============================] - 2s 9ms/step - loss: 0.1450 - accuracy: 0.9558 - val_loss: 0.1164 - val_accuracy: 0.9653
Epoch 5/50
188/188 [==============================] - 2s 9ms/step - loss: 0.1262 - accuracy: 0.9607 - val_loss: 0.1105 - val_accuracy: 0.9678
Epoch 6/50
188/188 [==============================] - 2s 10ms/step - loss: 0.1108 - accuracy: 0.9655 - val_loss: 0.1078 - val_accuracy: 0.9682
Epoch 7/50
188/188 [==============================] - 2s 9ms/step - loss: 0.1003 - accuracy: 0.9683 - val_loss: 0.1034 - val_accuracy: 0.9698
Epoc

In [19]:
#mnist fashion

(x_fashion_train, y_fashion_train), (x_fashion_test, y_fashion_test) = tf.keras.datasets.fashion_mnist.load_data()

selected_classes = [0, 1, 2, 3, 4]

train_mask = np.isin(y_fashion_train, selected_classes)
test_mask = np.isin(y_fashion_test, selected_classes)

X_fashion_train = x_fashion_train[train_mask]
y_fashion_train = y_fashion_train[train_mask]
X_fashion_test = x_fashion_test[test_mask]
y_fashion_test = y_fashion_test[test_mask]

X_fashion_train = X_fashion_train.reshape(X_fashion_train.shape[0], -1)
X_fashion_test = X_fashion_test.reshape(X_fashion_test.shape[0], -1)

X_fashion_train = scaler.transform(X_fashion_train)
X_fashion_test = scaler.transform(X_fashion_test)

In [20]:
fashion_model = Sequential()

for layer in model.layers[:-1]:
    fashion_model.add(layer)

for layer in fashion_model.layers:
    layer.trainable = False

fashion_model.add(
    Dense(32, activation='relu', kernel_initializer='he_normal')
)

fashion_model.add(BatchNormalization())

fashion_model.add(Dropout(0.2))

fashion_model.add(
    Dense(5, activation='softmax')
)

fashion_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

fashion_model.summary()

print("Обучение только новой головы на Fashion MNIST")

history_transfer = fashion_model.fit(
    X_fashion_train, y_fashion_train,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    verbose=1
)

y_pred_proba_fashion = fashion_model.predict(X_fashion_test, verbose=0)
y_pred_fashion = np.argmax(y_pred_proba_fashion, axis=1)

accuracy_transfer = np.mean(y_pred_fashion == y_fashion_test)

print(f"\nРезультаты Transfer Learning:")
print(f"Точность на тесте: {accuracy_transfer:.4f}")
print(
    classification_report(
        y_fashion_test,
        y_pred_fashion
    )
)

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_13 (Dense)            (None, 128)               100480    
                                                                 
 batch_normalization_8 (Batc  (None, 128)              512       
 hNormalization)                                                 
                                                                 
 dropout_8 (Dropout)         (None, 128)               0         
                                                                 
 dense_14 (Dense)            (None, 64)                8256      
                                                                 
 batch_normalization_9 (Batc  (None, 64)               256       
 hNormalization)                                                 
                                                                 
 dropout_9 (Dropout)         (None, 64)               

In [22]:
for layer in fashion_model.layers:
    layer.trainable = True

fashion_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = fashion_model.fit(
    X_fashion_train,
    y_fashion_train,

    validation_split=0.2,

    epochs=40,
    batch_size=256,

    verbose=1
)

y_pred_proba_fashion = fashion_model.predict(
    X_fashion_test,
    verbose=0
)

y_pred_fashion = np.argmax(
    y_pred_proba_fashion,
    axis=1
)

accuracy_finetune = np.mean(
    y_pred_fashion == y_fashion_test
)

print(
    f"Accuracy после разморозки тела: "
    f"{accuracy_finetune:.4f}"
)


print("\nClassification Report:")

print(
    classification_report(
        y_fashion_test,
        y_pred_fashion
    )
)

Epoch 1/40
94/94 [==============================] - 2s 15ms/step - loss: 0.4073 - accuracy: 0.8568 - val_loss: 0.3943 - val_accuracy: 0.8532
Epoch 2/40
94/94 [==============================] - 1s 12ms/step - loss: 0.3959 - accuracy: 0.8604 - val_loss: 0.3903 - val_accuracy: 0.8527
Epoch 3/40
94/94 [==============================] - 1s 13ms/step - loss: 0.3906 - accuracy: 0.8622 - val_loss: 0.3773 - val_accuracy: 0.8607
Epoch 4/40
94/94 [==============================] - 1s 12ms/step - loss: 0.3917 - accuracy: 0.8630 - val_loss: 0.3831 - val_accuracy: 0.8558
Epoch 5/40
94/94 [==============================] - 1s 12ms/step - loss: 0.3830 - accuracy: 0.8658 - val_loss: 0.3694 - val_accuracy: 0.8603
Epoch 6/40
94/94 [==============================] - 1s 13ms/step - loss: 0.3783 - accuracy: 0.8654 - val_loss: 0.3790 - val_accuracy: 0.8520
Epoch 7/40
94/94 [==============================] - 1s 12ms/step - loss: 0.3733 - accuracy: 0.8693 - val_loss: 0.3637 - val_accuracy: 0.8622
Epoch 8/40
94